# Traditional ML Modelling — Literature-Motivated Main Specification

This notebook reruns:
- Logistic Regression
- Random Forest
- XGBoost

using the corrected model-ready dataset.

## Baseline
- `Lag1_Return`
- `Lag2_Return`
- `Lag3_Return`

## Main Enhanced specification

The Baseline plus:
- `PM_Prob_Lag1`
- `PM_Change_1D_Lag1`
- `PM_Prob_x_TimeToResolution`
- `PM_Log_Cumulative_Support_Notional_Lag1`
- `PM_Net_Support_Imbalance_Lag1`
- `Days_to_Resolution_Years`

The first five variables represent the same literature-motivated Polymarket information channels used in Conditional Logit.

`Days_to_Resolution_Years` is additionally allowed in traditional ML because event-constant predictors do not cancel in the same way as Conditional Logit.

Other corrected activity variables remain in the RQ2 feature-ablation / robustness analysis rather than being selected from final-test performance.


## Step 1 — Upload the Corrected Model-Ready Excel File

Upload:

`Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features.xlsx`

A downloaded copy containing `(1)` is also accepted.


## Step 2 — Install and Import Packages


In [ ]:
# ============================================================
# STEP 2: Install and Import Packages
# ============================================================

!pip -q install xgboost openpyxl statsmodels scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBClassifier

import statsmodels.api as sm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Packages imported successfully.")


## Step 3 — Load the Model-Ready Dataset


In [ ]:
# ============================================================
# STEP 3: Load Corrected Model-Ready Master Sheet
# ============================================================

import glob
import os

candidates = glob.glob(
    "/content/*CORRECTED_PM_Trading_Features*.xlsx"
)

dataset_candidates = [
    x for x in candidates
    if "MODEL_READY" in os.path.basename(x).upper()
]

if len(dataset_candidates) != 1:
    raise ValueError(
        "Upload exactly one corrected model-ready dataset. "
        f"Detected: {dataset_candidates}"
    )

FILE_PATH = dataset_candidates[0]

df = pd.read_excel(
    FILE_PATH,
    sheet_name="Model_Ready_Master"
)

date_columns = [
    "Date",
    "Next_Financial_Date",
    "Financial_Prev1_Date",
    "Financial_Prev2_Date",
    "Financial_Prev3_Date",
    "PM_Signal_Date_Lag1",
    "PM_Refined_Activity_Signal_Date_Lag1"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

df = df.sort_values(
    ["Date", "Market_ID", "Event_ID", "Asset"]
).reset_index(drop=True)

print("Loaded:", FILE_PATH)
print("Rows:", len(df))
print("Event IDs:", df["Event_ID"].nunique())
print("Unique dates:", df["Date"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())

print("\nEvents by market:")
print(
    df.groupby("Market_ID")["Event_ID"].nunique()
)

display(df.head())


## Step 4 — Define the Baseline and New Enhanced Features

The main Enhanced model uses the original probability information plus a compact corrected trading-activity core. The wider activity set is analysed separately in the RQ2 ablation.


In [ ]:
BASELINE_FEATURES = [
    "Lag1_Return",
    "Lag2_Return",
    "Lag3_Return"
]

PM_MAIN_FEATURES = [
    "PM_Prob_Lag1",
    "PM_Change_1D_Lag1",
    "PM_Prob_x_TimeToResolution",
    "PM_Log_Cumulative_Support_Notional_Lag1",
    "PM_Net_Support_Imbalance_Lag1"
]

ENHANCED_FEATURES = (
    BASELINE_FEATURES
    + PM_MAIN_FEATURES
    + ["Days_to_Resolution_Years"]
)

RQ2_ONLY_FEATURES = [
    "PM_Change_From_Start_Lag1",
    "PM_Log_Support_Notional_Lag1",
    "PM_Relative_Support_Volume_Share_Lag1",
    "PM_Relative_Cumulative_Support_Share_Lag1",
    "PM_Log_Support_Unique_Takers_Lag1",
    "PM_Support_Volume_Surge_Lag1",
    "PM_Prob_x_Log_Contract_Notional_Lag1",
    "PM_Support_HHI_Lag1",
    "PM_Log_Support_Mean_Trade_Notional_Lag1",
    "PM_Candidate_Price_Dispersion_Lag1"
]

NEVER_USE_AS_PREDICTORS = [
    "Winner",
    "Forward_Log_Return",
    "Price_Next",
    "PM_Prob_Current_Audit"
]

feature_set_table = pd.DataFrame([
    {
        "Specification": "Baseline",
        "Model_Family": "Traditional ML",
        "Features": " + ".join(BASELINE_FEATURES)
    },
    {
        "Specification": "Enhanced",
        "Model_Family": "Traditional ML",
        "Features": " + ".join(ENHANCED_FEATURES)
    }
])

print("BASELINE FEATURES:")
print(BASELINE_FEATURES)

print("\nLITERATURE-MOTIVATED MAIN POLYMARKET FEATURES:")
print(PM_MAIN_FEATURES)

print("\nMAIN ENHANCED FEATURES:")
print(ENHANCED_FEATURES)

print("\nRQ2 / ROBUSTNESS-ONLY FEATURES:")
print(RQ2_ONLY_FEATURES)


## Step 5 — Final Leakage and Structure Checks

These checks confirm that all modelling variables are known before the future winner is observed.


In [ ]:
# ============================================================
# STEP 5: Leakage and Structure Checks
# ============================================================

winner_counts = df.groupby("Event_ID")["Winner"].sum()
assert (winner_counts == 1).all(),     "ERROR: Some Event IDs do not contain exactly one winner."

choice_sizes = df.groupby("Event_ID")["Asset"].nunique()
assert choice_sizes.isin([2, 3]).all(),     "ERROR: Unexpected choice-set size."

assert (
    df["PM_Signal_Date_Lag1"] < df["Date"]
).all(), "ERROR: Polymarket timing leakage detected."

assert (
    df["PM_Refined_Activity_Signal_Date_Lag1"] < df["Date"]
).all(), "ERROR: Corrected PM trading-activity timing leakage detected."

assert (
    df["Financial_Prev1_Date"] < df["Date"]
).all(), "ERROR: Financial lag timing leakage detected."

required = list(dict.fromkeys(
    BASELINE_FEATURES + ENHANCED_FEATURES
))

assert df[required].notna().all().all(),     "ERROR: Missing modelling predictors detected."

assert len(df) == 1012, "ERROR: Corrected master should contain 1,012 rows."
assert df["Event_ID"].nunique() == 428, "ERROR: Corrected master should contain 428 events."

print("All checks PASSED — including corrected PM trading timing.")

print("\nWinner count per event:")
print(winner_counts.value_counts().sort_index())

print("\nChoice-set sizes:")
print(choice_sizes.value_counts().sort_index())


## Step 6 — Freeze the Same Chronological 80/20 Split

The split is created from unique dates so all competitions on the same date stay in the same partition.


In [ ]:
# ============================================================
# STEP 6: Freeze Development and Final Test Dates
# ============================================================

unique_dates = np.array(
    sorted(df["Date"].dropna().unique())
)

n_dates = len(unique_dates)
n_dev_dates = int(np.floor(0.80 * n_dates))

dev_dates = unique_dates[:n_dev_dates]
test_dates = unique_dates[n_dev_dates:]

dev = df[
    df["Date"].isin(dev_dates)
].copy()

final_test = df[
    df["Date"].isin(test_dates)
].copy()

print("Total unique dates:", n_dates)
print("Development dates:", len(dev_dates))
print("Final test dates:", len(test_dates))

print("\nDEVELOPMENT:")
print("Dates:", dev["Date"].min(), "to", dev["Date"].max())
print("Events:", dev["Event_ID"].nunique())
print("Rows:", len(dev))

print("\nFINAL TEST:")
print("Dates:", final_test["Date"].min(), "to", final_test["Date"].max())
print("Events:", final_test["Event_ID"].nunique())
print("Rows:", len(final_test))

print("\nFinal-test events by market:")
print(
    final_test.groupby("Market_ID")["Event_ID"].nunique()
)


## Step 7 — Create the Same Four Expanding-Window Validation Folds

These folds match the Conditional Logit evaluation design.


In [ ]:
# ============================================================
# STEP 7: Create Expanding-Window Validation Folds
# ============================================================

tscv = TimeSeriesSplit(
    n_splits=4,
    test_size=25,
    gap=0
)

cv_folds = []

for fold_no, (train_idx, val_idx) in enumerate(
    tscv.split(dev_dates),
    start=1
):
    train_dates_fold = dev_dates[train_idx]
    val_dates_fold = dev_dates[val_idx]

    train_fold = df[
        df["Date"].isin(train_dates_fold)
    ].copy()

    val_fold = df[
        df["Date"].isin(val_dates_fold)
    ].copy()

    assert train_fold["Date"].max() < val_fold["Date"].min()
    assert set(train_fold["Date"]).isdisjoint(set(val_fold["Date"]))
    assert set(train_fold["Event_ID"]).isdisjoint(set(val_fold["Event_ID"]))

    cv_folds.append({
        "Fold": fold_no,
        "Train_Dates": train_dates_fold,
        "Val_Dates": val_dates_fold
    })

    print(f"\nFOLD {fold_no}")
    print(
        "Train:",
        train_fold["Date"].min().date(),
        "to",
        train_fold["Date"].max().date(),
        "| Events:",
        train_fold["Event_ID"].nunique()
    )
    print(
        "Validation:",
        val_fold["Date"].min().date(),
        "to",
        val_fold["Date"].max().date(),
        "| Events:",
        val_fold["Event_ID"].nunique()
    )


## Step 8 — Helper Functions

The conventional ML models treat each alternative row as a binary winner/non-winner observation.

For evaluation, each row-level predicted winner probability is normalized **within Event ID** so that the competing alternatives form one probability distribution that sums to 1.

The highest event-level probability is the predicted winner.


In [ ]:
# ============================================================
# STEP 8A: Normalize Row Probabilities Within Each Event
# ============================================================

def make_event_predictions(model, test_df, features):
    row_prob = model.predict_proba(
        test_df[features]
    )[:, 1]

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner"
        ]
    ].copy()

    pred["Raw_Winner_Prob"] = row_prob
    pred["Pred_Prob"] = np.nan

    for event_id, idx in pred.groupby("Event_ID").groups.items():
        idx = list(idx)

        probs = pred.loc[
            idx,
            "Raw_Winner_Prob"
        ].to_numpy(dtype=float)

        # Safety against zero sums
        probs = np.clip(probs, 1e-15, None)
        probs = probs / probs.sum()

        pred.loc[idx, "Pred_Prob"] = probs

    return pred


In [ ]:
# ============================================================
# STEP 8B: Event-Level Evaluation Metrics
# ============================================================

def evaluate_event_predictions(pred):
    eps = 1e-15

    event_rows = []

    for event_id, g in pred.groupby("Event_ID"):
        g = g.copy()

        probs = g["Pred_Prob"].to_numpy(dtype=float)
        probs = probs / probs.sum()

        y = g["Winner"].to_numpy(dtype=int)

        actual_idx = np.where(y == 1)[0][0]
        predicted_idx = int(np.argmax(probs))

        actual_winner = g.iloc[actual_idx]["Asset"]
        predicted_winner = g.iloc[predicted_idx]["Asset"]

        winner_prob = float(
            np.clip(probs[actual_idx], eps, 1 - eps)
        )

        log_loss_event = -np.log(winner_prob)

        # Same multiclass Brier definition used in the Conditional Logit notebook
        brier_event = np.sum(
            (probs - y) ** 2
        )

        correct = int(
            actual_winner == predicted_winner
        )

        J = len(g)

        event_rows.append({
            "Market_ID": g["Market_ID"].iloc[0],
            "Event_ID": event_id,
            "Date": g["Date"].iloc[0],
            "Choice_Set_Size": J,
            "Actual_Winner": actual_winner,
            "Predicted_Winner": predicted_winner,
            "Winner_Probability": winner_prob,
            "Correct": correct,
            "Log_Loss": log_loss_event,
            "Brier_Score": brier_event,
            "Uniform_Log_Loss": np.log(J),
            "Uniform_Brier": 1 - (1 / J)
        })

    event_metrics = pd.DataFrame(event_rows)

    summary = {
        "Events": len(event_metrics),
        "Log_Loss": event_metrics["Log_Loss"].mean(),
        "Brier_Score": event_metrics["Brier_Score"].mean(),
        "Accuracy": event_metrics["Correct"].mean(),
        "Chance_Accuracy": (
            1 / event_metrics["Choice_Set_Size"]
        ).mean(),
        "Log_Skill_vs_Uniform": 1 - (
            event_metrics["Log_Loss"].sum() /
            event_metrics["Uniform_Log_Loss"].sum()
        ),
        "Brier_Skill_vs_Uniform": 1 - (
            event_metrics["Brier_Score"].sum() /
            event_metrics["Uniform_Brier"].sum()
        )
    }

    return summary, event_metrics


## Step 9 — Define Small Hyperparameter Grids

The grids are deliberately modest because the dataset contains only about 428 competitive events.

Hyperparameters are chosen using the mean expanding-window **Log Loss** across the four validation folds.

This avoids over-tuning the small sample.


In [ ]:
# ============================================================
# STEP 9: Define Hyperparameter Grids
# ============================================================

MODEL_GRIDS = {
    "Logistic_Regression": [
        {"C": 0.1},
        {"C": 1.0},
        {"C": 10.0}
    ],

    "Random_Forest": [
        {
            "n_estimators": 300,
            "max_depth": 3,
            "min_samples_leaf": 5,
            "max_features": "sqrt"
        },
        {
            "n_estimators": 300,
            "max_depth": 5,
            "min_samples_leaf": 5,
            "max_features": "sqrt"
        },
        {
            "n_estimators": 500,
            "max_depth": None,
            "min_samples_leaf": 5,
            "max_features": "sqrt"
        }
    ],

    "XGBoost": [
        {
            "n_estimators": 150,
            "max_depth": 2,
            "learning_rate": 0.03,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        },
        {
            "n_estimators": 200,
            "max_depth": 3,
            "learning_rate": 0.03,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        },
        {
            "n_estimators": 150,
            "max_depth": 3,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        }
    ]
}

for model_name, grid in MODEL_GRIDS.items():
    print(model_name, "candidate settings:", len(grid))


## Step 10 — Model Builder

No class balancing is used because the primary objective is calibrated probability forecasting rather than only classification accuracy.


In [ ]:
# ============================================================
# STEP 10: Build Models
# ============================================================

def build_model(model_name, params):

    if model_name == "Logistic_Regression":

        return Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    C=params["C"],
                    penalty="l2",
                    solver="lbfgs",
                    max_iter=2000,
                    random_state=42
                )
            )
        ])

    elif model_name == "Random_Forest":

        return RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "XGBoost":

        return XGBClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0
        )

    else:
        raise ValueError("Unknown model name.")


## Step 11 — Expanding-Window Hyperparameter Selection

Each candidate model is evaluated on the same four temporal folds.

The best setting is the one with the lowest mean validation Log Loss.


In [ ]:
# ============================================================
# STEP 11: Tune Models Using Expanding-Window Validation
# ============================================================

tuning_rows = []
fold_prediction_tables = []

for model_name, param_grid in MODEL_GRIDS.items():

    print("\n" + "=" * 80)
    print("TUNING:", model_name)
    print("=" * 80)

    for specification, features in [
        ("Baseline", BASELINE_FEATURES),
        ("Enhanced", ENHANCED_FEATURES)
    ]:

        print(f"\n--- {specification} ---")

        for candidate_no, params in enumerate(
            param_grid,
            start=1
        ):

            fold_metrics = []

            for fold_info in cv_folds:

                train_fold = df[
                    df["Date"].isin(
                        fold_info["Train_Dates"]
                    )
                ].copy()

                val_fold = df[
                    df["Date"].isin(
                        fold_info["Val_Dates"]
                    )
                ].copy()

                model = build_model(
                    model_name,
                    params
                )

                model.fit(
                    train_fold[features],
                    train_fold["Winner"].astype(int)
                )

                pred = make_event_predictions(
                    model,
                    val_fold,
                    features
                )

                metrics, event_metrics = (
                    evaluate_event_predictions(pred)
                )

                fold_metrics.append(metrics)

                event_metrics["Model"] = model_name
                event_metrics["Specification"] = specification
                event_metrics["Candidate"] = candidate_no
                event_metrics["Fold"] = fold_info["Fold"]
                fold_prediction_tables.append(event_metrics)

            mean_log_loss = np.mean([
                x["Log_Loss"]
                for x in fold_metrics
            ])

            mean_brier = np.mean([
                x["Brier_Score"]
                for x in fold_metrics
            ])

            mean_accuracy = np.mean([
                x["Accuracy"]
                for x in fold_metrics
            ])

            tuning_rows.append({
                "Model": model_name,
                "Specification": specification,
                "Candidate": candidate_no,
                "Parameters": str(params),
                "Mean_CV_Log_Loss": mean_log_loss,
                "Mean_CV_Brier": mean_brier,
                "Mean_CV_Accuracy": mean_accuracy
            })

            print(
                f"Candidate {candidate_no}: "
                f"Log Loss={mean_log_loss:.4f} | "
                f"Brier={mean_brier:.4f} | "
                f"Accuracy={mean_accuracy:.3f}"
            )

tuning_results = pd.DataFrame(tuning_rows)

cv_all_event_predictions = pd.concat(
    fold_prediction_tables,
    ignore_index=True
)

display(
    tuning_results.sort_values(
        ["Model", "Specification", "Mean_CV_Log_Loss"]
    )
)


## Step 12 — Select the Best Hyperparameters

Selection is based only on validation Log Loss.


In [ ]:
# ============================================================
# STEP 12: Select Best Hyperparameters
# ============================================================

best_rows = (
    tuning_results
    .sort_values("Mean_CV_Log_Loss")
    .groupby(
        ["Model", "Specification"],
        as_index=False
    )
    .first()
)

display(best_rows)

# Reconstruct the original dictionaries
best_params = {}

for _, row in best_rows.iterrows():

    model_name = row["Model"]
    specification = row["Specification"]
    candidate = int(row["Candidate"])

    best_params[
        (model_name, specification)
    ] = MODEL_GRIDS[model_name][candidate - 1]

print("\nSelected settings:")
for key, value in best_params.items():
    print(key, "->", value)


## Step 13 — Extract Validation Results for the Selected Models Only

This shows the fold-by-fold performance of the final selected specification for each model.


In [ ]:
# ============================================================
# STEP 13: Selected Expanding-Window CV Results
# ============================================================

selected_cv_rows = []
selected_cv_events = []

for model_name in MODEL_GRIDS.keys():

    for specification, features in [
        ("Baseline", BASELINE_FEATURES),
        ("Enhanced", ENHANCED_FEATURES)
    ]:

        params = best_params[
            (model_name, specification)
        ]

        for fold_info in cv_folds:

            train_fold = df[
                df["Date"].isin(
                    fold_info["Train_Dates"]
                )
            ].copy()

            val_fold = df[
                df["Date"].isin(
                    fold_info["Val_Dates"]
                )
            ].copy()

            model = build_model(
                model_name,
                params
            )

            model.fit(
                train_fold[features],
                train_fold["Winner"].astype(int)
            )

            pred = make_event_predictions(
                model,
                val_fold,
                features
            )

            metrics, event_metrics = (
                evaluate_event_predictions(pred)
            )

            metrics.update({
                "Model": model_name,
                "Specification": specification,
                "Fold": fold_info["Fold"]
            })

            selected_cv_rows.append(metrics)

            event_metrics["Model"] = model_name
            event_metrics["Specification"] = specification
            event_metrics["Fold"] = fold_info["Fold"]
            selected_cv_events.append(event_metrics)

selected_cv_results = pd.DataFrame(
    selected_cv_rows
)

selected_cv_event_predictions = pd.concat(
    selected_cv_events,
    ignore_index=True
)

display(
    selected_cv_results.sort_values(
        ["Model", "Fold", "Specification"]
    )
)


## Step 14 — Summarize Selected Cross-Validation Results

The key comparison remains:

- Negative Enhanced − Baseline Log Loss = Polymarket improvement
- Negative Enhanced − Baseline Brier = Polymarket improvement
- Positive Enhanced − Baseline Accuracy = Polymarket improvement


In [ ]:
# ============================================================
# STEP 14: Cross-Validation Summary and Differences
# ============================================================

cv_summary = (
    selected_cv_results
    .groupby(
        ["Model", "Specification"]
    )
    .agg(
        Mean_Log_Loss=("Log_Loss", "mean"),
        SD_Log_Loss=("Log_Loss", "std"),
        Mean_Brier=("Brier_Score", "mean"),
        SD_Brier=("Brier_Score", "std"),
        Mean_Accuracy=("Accuracy", "mean"),
        SD_Accuracy=("Accuracy", "std"),
        Mean_Log_Skill=("Log_Skill_vs_Uniform", "mean"),
        Mean_Brier_Skill=("Brier_Skill_vs_Uniform", "mean")
    )
    .reset_index()
)

display(cv_summary)

cv_difference_rows = []

for model_name in selected_cv_results["Model"].unique():

    temp = selected_cv_results[
        selected_cv_results["Model"] == model_name
    ]

    wide = temp.pivot(
        index="Fold",
        columns="Specification",
        values=[
            "Log_Loss",
            "Brier_Score",
            "Accuracy"
        ]
    )

    for fold in wide.index:

        cv_difference_rows.append({
            "Model": model_name,
            "Fold": fold,
            "LogLoss_Enhanced_minus_Baseline":
                wide.loc[fold, ("Log_Loss", "Enhanced")] -
                wide.loc[fold, ("Log_Loss", "Baseline")],
            "Brier_Enhanced_minus_Baseline":
                wide.loc[fold, ("Brier_Score", "Enhanced")] -
                wide.loc[fold, ("Brier_Score", "Baseline")],
            "Accuracy_Enhanced_minus_Baseline":
                wide.loc[fold, ("Accuracy", "Enhanced")] -
                wide.loc[fold, ("Accuracy", "Baseline")]
        })

cv_differences = pd.DataFrame(
    cv_difference_rows
)

display(cv_differences)


## Step 15 — Train the Selected Models on the Full Development Period and Evaluate the Untouched Final Test

This is the key out-of-sample result for the dissertation.


In [ ]:
# ============================================================
# STEP 15: Final Untouched Test
# ============================================================

final_results = []
final_event_predictions = []
final_row_predictions = []
fitted_models = {}

for model_name in MODEL_GRIDS.keys():

    for specification, features in [
        ("Baseline", BASELINE_FEATURES),
        ("Enhanced", ENHANCED_FEATURES)
    ]:

        params = best_params[
            (model_name, specification)
        ]

        model = build_model(
            model_name,
            params
        )

        model.fit(
            dev[features],
            dev["Winner"].astype(int)
        )

        pred = make_event_predictions(
            model,
            final_test,
            features
        )

        metrics, event_metrics = (
            evaluate_event_predictions(pred)
        )

        metrics.update({
            "Model": model_name,
            "Specification": specification,
            "Parameters": str(params)
        })

        final_results.append(metrics)

        event_metrics["Model"] = model_name
        event_metrics["Specification"] = specification
        final_event_predictions.append(event_metrics)

        pred["Model"] = model_name
        pred["Specification"] = specification
        final_row_predictions.append(pred)

        fitted_models[
            (model_name, specification)
        ] = model

final_results_df = pd.DataFrame(
    final_results
)

final_event_predictions_df = pd.concat(
    final_event_predictions,
    ignore_index=True
)

final_row_predictions_df = pd.concat(
    final_row_predictions,
    ignore_index=True
)

display(
    final_results_df.sort_values(
        ["Model", "Specification"]
    )
)


## Step 16 — Calculate Final Baseline vs Enhanced Differences

This table directly answers whether Polymarket improved each ML model.


In [ ]:
# ============================================================
# STEP 16: Final Enhanced vs Baseline Differences
# ============================================================

final_difference_rows = []

for model_name in final_results_df["Model"].unique():

    temp = final_results_df[
        final_results_df["Model"] == model_name
    ].set_index("Specification")

    final_difference_rows.append({
        "Model": model_name,

        "Baseline_Log_Loss":
            temp.loc["Baseline", "Log_Loss"],
        "Enhanced_Log_Loss":
            temp.loc["Enhanced", "Log_Loss"],
        "LogLoss_Enhanced_minus_Baseline":
            temp.loc["Enhanced", "Log_Loss"] -
            temp.loc["Baseline", "Log_Loss"],

        "Baseline_Brier":
            temp.loc["Baseline", "Brier_Score"],
        "Enhanced_Brier":
            temp.loc["Enhanced", "Brier_Score"],
        "Brier_Enhanced_minus_Baseline":
            temp.loc["Enhanced", "Brier_Score"] -
            temp.loc["Baseline", "Brier_Score"],

        "Baseline_Accuracy":
            temp.loc["Baseline", "Accuracy"],
        "Enhanced_Accuracy":
            temp.loc["Enhanced", "Accuracy"],
        "Accuracy_Enhanced_minus_Baseline":
            temp.loc["Enhanced", "Accuracy"] -
            temp.loc["Baseline", "Accuracy"]
    })

final_differences = pd.DataFrame(
    final_difference_rows
)

print(
    "Negative Log Loss / Brier difference = Enhanced better."
)
print(
    "Positive Accuracy difference = Enhanced better."
)

display(final_differences)


## Step 17 — Final Results by Market

This is necessary because:

- BTC–Gold events have 2 alternatives and a 50% uniform chance benchmark.
- BTC–Gold–S&P events have 3 alternatives and a 33.3% uniform chance benchmark.

Overall results should therefore be supplemented with market-specific performance.


In [ ]:
# ============================================================
# STEP 17: Final Performance by Market
# ============================================================

market_rows = []

for (
    model_name,
    specification,
    market_id
), g in final_event_predictions_df.groupby(
    ["Model", "Specification", "Market_ID"]
):

    market_rows.append({
        "Model": model_name,
        "Specification": specification,
        "Market_ID": market_id,
        "Events": len(g),
        "Log_Loss": g["Log_Loss"].mean(),
        "Brier_Score": g["Brier_Score"].mean(),
        "Accuracy": g["Correct"].mean(),
        "Chance_Accuracy": (
            1 / g["Choice_Set_Size"]
        ).mean()
    })

final_by_market = pd.DataFrame(
    market_rows
)

display(
    final_by_market.sort_values(
        ["Model", "Market_ID", "Specification"]
    )
)


## Step 18 — Feature Importance / Coefficients for the Enhanced Models

These are supporting interpretation for RQ2: which types of Polymarket information appear most useful?

They are descriptive and should not replace out-of-sample predictive evaluation.


In [ ]:
# ============================================================
# STEP 18: Extract Enhanced Model Feature Importance
# ============================================================

importance_rows = []

# Logistic Regression standardized coefficients
log_model = fitted_models[
    ("Logistic_Regression", "Enhanced")
]

log_coef = (
    log_model.named_steps["model"].coef_[0]
)

for feature, value in zip(
    ENHANCED_FEATURES,
    log_coef
):
    importance_rows.append({
        "Model": "Logistic_Regression",
        "Feature": feature,
        "Importance_or_Coefficient": value,
        "Type": "Standardized coefficient"
    })

# Random Forest feature importance
rf_model = fitted_models[
    ("Random_Forest", "Enhanced")
]

for feature, value in zip(
    ENHANCED_FEATURES,
    rf_model.feature_importances_
):
    importance_rows.append({
        "Model": "Random_Forest",
        "Feature": feature,
        "Importance_or_Coefficient": value,
        "Type": "Impurity importance"
    })

# XGBoost feature importance
xgb_model = fitted_models[
    ("XGBoost", "Enhanced")
]

for feature, value in zip(
    ENHANCED_FEATURES,
    xgb_model.feature_importances_
):
    importance_rows.append({
        "Model": "XGBoost",
        "Feature": feature,
        "Importance_or_Coefficient": value,
        "Type": "XGBoost feature importance"
    })

feature_importance = pd.DataFrame(
    importance_rows
)

display(
    feature_importance.sort_values(
        ["Model", "Importance_or_Coefficient"],
        ascending=[True, False]
    )
)


## Step 19 — Date-Level Statistical Comparison of Baseline vs Enhanced

Because multiple competitions can occur on the same date and share underlying financial outcomes, loss differences are aggregated to the **calendar-date level** before the HAC comparison.

This is supporting statistical evidence.


In [ ]:
# ============================================================
# STEP 19A: Create Date-Level Loss Differences
# ============================================================

date_difference_rows = []

for model_name in final_event_predictions_df["Model"].unique():

    temp = final_event_predictions_df[
        final_event_predictions_df["Model"] == model_name
    ]

    date_model = (
        temp.groupby(
            ["Date", "Specification"]
        )
        .agg(
            Mean_Log_Loss=("Log_Loss", "mean"),
            Mean_Brier=("Brier_Score", "mean"),
            Mean_Accuracy=("Correct", "mean")
        )
        .reset_index()
    )

    wide = date_model.pivot(
        index="Date",
        columns="Specification",
        values=[
            "Mean_Log_Loss",
            "Mean_Brier",
            "Mean_Accuracy"
        ]
    )

    for date in wide.index:

        date_difference_rows.append({
            "Model": model_name,
            "Date": date,
            "LogLoss_Baseline_minus_Enhanced":
                wide.loc[date, ("Mean_Log_Loss", "Baseline")] -
                wide.loc[date, ("Mean_Log_Loss", "Enhanced")],
            "Brier_Baseline_minus_Enhanced":
                wide.loc[date, ("Mean_Brier", "Baseline")] -
                wide.loc[date, ("Mean_Brier", "Enhanced")],
            "Accuracy_Enhanced_minus_Baseline":
                wide.loc[date, ("Mean_Accuracy", "Enhanced")] -
                wide.loc[date, ("Mean_Accuracy", "Baseline")]
        })

date_loss_differences = pd.DataFrame(
    date_difference_rows
)

display(date_loss_differences.head())


In [ ]:
# ============================================================
# STEP 19B: HAC Mean-Difference Tests
# ============================================================

def hac_mean_test(series, maxlags=5):
    x = pd.Series(series).dropna().astype(float)

    X = np.ones((len(x), 1))

    result = sm.OLS(
        x.to_numpy(),
        X
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags}
    )

    return {
        "Mean_Difference": result.params[0],
        "HAC_Std_Error": result.bse[0],
        "t_Statistic": result.tvalues[0],
        "p_Value": result.pvalues[0],
        "N_Dates": len(x),
        "HAC_MaxLags": maxlags
    }

hac_rows = []

for model_name, g in date_loss_differences.groupby("Model"):

    log_test = hac_mean_test(
        g["LogLoss_Baseline_minus_Enhanced"],
        maxlags=5
    )

    brier_test = hac_mean_test(
        g["Brier_Baseline_minus_Enhanced"],
        maxlags=5
    )

    hac_rows.append({
        "Model": model_name,
        "Metric": "Log Loss",
        **log_test
    })

    hac_rows.append({
        "Model": model_name,
        "Metric": "Brier Score",
        **brier_test
    })

hac_tests = pd.DataFrame(
    hac_rows
)

print(
    "Positive mean loss difference = Enhanced has lower loss."
)

display(hac_tests)


## Step 20 — Create a Compact Felipe Summary Table

This table is designed for a quick supervisor discussion.

The main question is whether Enhanced improves the primary Log Loss metric and whether the direction is supported by Accuracy/Brier.


In [ ]:
# ============================================================
# STEP 20: Compact Baseline vs Enhanced Summary
# ============================================================

felipe_rows = []

for _, r in final_differences.iterrows():

    log_result = (
        "Improved"
        if r["LogLoss_Enhanced_minus_Baseline"] < 0
        else "Worse"
    )

    brier_result = (
        "Improved"
        if r["Brier_Enhanced_minus_Baseline"] < 0
        else "Worse"
    )

    acc_result = (
        "Improved"
        if r["Accuracy_Enhanced_minus_Baseline"] > 0
        else "No improvement"
    )

    felipe_rows.append({
        "Model": r["Model"],
        "Baseline Log Loss": r["Baseline_Log_Loss"],
        "Enhanced Log Loss": r["Enhanced_Log_Loss"],
        "Log Loss Result": log_result,
        "Baseline Accuracy": r["Baseline_Accuracy"],
        "Enhanced Accuracy": r["Enhanced_Accuracy"],
        "Accuracy Result": acc_result,
        "Brier Result": brier_result
    })

felipe_summary = pd.DataFrame(
    felipe_rows
)

display(felipe_summary)


## Step 21 — Save All ML Results to Excel

The workbook contains:

- Hyperparameter tuning results
- Selected hyperparameters
- Selected CV fold results
- CV summary
- CV baseline/enhanced differences
- Final overall test results
- Final baseline/enhanced differences
- Final market-specific results
- Final event-level predictions
- Final row-level probabilities
- Feature importance / coefficients
- Date-level loss differences
- HAC loss comparison tests
- Compact supervisor summary


In [ ]:
# ============================================================
# STEP 21: Save Results to Excel
# ============================================================

OUTPUT_FILE = (
    "/content/"
    "Traditional_ML_Final_Evaluation_All_Results.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    tuning_results.to_excel(
        writer,
        sheet_name="Tuning_All",
        index=False
    )

    best_rows.to_excel(
        writer,
        sheet_name="Best_Hyperparameters",
        index=False
    )

    selected_cv_results.to_excel(
        writer,
        sheet_name="CV_Fold_Results",
        index=False
    )

    cv_summary.to_excel(
        writer,
        sheet_name="CV_Summary",
        index=False
    )

    cv_differences.to_excel(
        writer,
        sheet_name="CV_Differences",
        index=False
    )

    selected_cv_event_predictions.to_excel(
        writer,
        sheet_name="CV_Event_Predictions",
        index=False
    )

    final_results_df.to_excel(
        writer,
        sheet_name="Final_Test_Overall",
        index=False
    )

    final_differences.to_excel(
        writer,
        sheet_name="Final_Differences",
        index=False
    )

    final_by_market.to_excel(
        writer,
        sheet_name="Final_Test_By_Market",
        index=False
    )

    final_event_predictions_df.to_excel(
        writer,
        sheet_name="Final_Event_Predictions",
        index=False
    )

    final_row_predictions_df.to_excel(
        writer,
        sheet_name="Final_Row_Probabilities",
        index=False
    )

    feature_importance.to_excel(
        writer,
        sheet_name="Feature_Importance",
        index=False
    )

    date_loss_differences.to_excel(
        writer,
        sheet_name="Date_Loss_Differences",
        index=False
    )

    hac_tests.to_excel(
        writer,
        sheet_name="HAC_Loss_Tests",
        index=False
    )

    felipe_summary.to_excel(
        writer,
        sheet_name="Felipe_Summary",
        index=False
    )

    feature_set_table.to_excel(
        writer,
        sheet_name="Feature_Set_Definitions",
        index=False
    )

print("Saved successfully:")
print(OUTPUT_FILE)


## Step 22 — Download the ML Results


In [ ]:
# ============================================================
# STEP 22: Download Results Workbook
# ============================================================

from google.colab import files

files.download(
    "/content/"
    "Traditional_ML_Final_Evaluation_All_Results.xlsx"
)
